# GCS Sharding Dataset con PyTorch

Questo notebook carica il dataset Iris da Google Cloud Storage (GCS), lo divide in shard (frammenti), e crea un DataLoader lazy che scarica i dati on-demand dai shard.

## Workflow
1. **Autenticazione** → Login con credenziali Google
2. **Accesso al bucket** → Leggi file da GCS
3. **Preprocessing** → Carica CSV e converti in tensori PyTorch
4. **Sharding** → Dividi il dataset e carica su GCS
5. **Dataset Lazy** → Leggi shard su richiesta durante l'addestramento

## Step 1: Autenticazione con Google Cloud

Autentica il tuo account Google per accedere ai bucket GCS con le credenziali locali salvate da `gcloud`.

In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=7yc5PGhRYKzVhXenUEzRMlP6EdwLm9&access_type=offline&code_challenge=_UJ_ag9QzXstsaXzFIBviNoE6LEnTIwCE8S7gVAIZ3s&code_challenge_method=S256


Credentials saved to file: [/Users/lgu/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "test-project-493915" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


## Step 2: Configurazione del Bucket GCS

Specifica il nome del bucket GCS dove sono salvati i dati.

In [2]:
NOME_DEL_BUCKET="ias-luigi-asprino-bucket"

## Step 3: Accesso al Bucket con gcsfs

`gcsfs` usa le credenziali autenticate (`google_default`) per elencare i file nel bucket.

In [3]:
# Cella 2: accesso al bucket con gcsfs
import gcsfs

fs = gcsfs.GCSFileSystem(token="google_default")
files = fs.ls(f'gs://{NOME_DEL_BUCKET}/')   # lista gli oggetti
print(files)

['ias-luigi-asprino-bucket/iris.csv', 'ias-luigi-asprino-bucket/shards']


## Step 4: Caricamento del Dataset

Leggi il file CSV `iris.csv` dal bucket usando `gcsfs.open()` e carica in un pandas DataFrame.

In [4]:
import pandas, io
# Leggere un CSV
with fs.open(f'gs://{NOME_DEL_BUCKET}/iris.csv', 'rb') as f:
    df = pandas.read_csv(io.BytesIO(f.read()))

print(f"Dataset caricato: {len(df)} righe, colonne: {list(df.columns)}")


Dataset caricato: 150 righe, colonne: ['sepal.length', 'sepal.width', 'petal.length', 'petal.width', 'variety']


## Step 5: Analisi Esplorativa

Analizza i tipi di dato, colonne categoriche, valori mancanti e visualizza le prime righe.

In [5]:
# Controlla i tipi di ogni colonna
print(df.dtypes)

# Identifica le colonne object
print(df.select_dtypes(include='object').columns.tolist())

# Controlla i valori mancanti
print(df.isnull().sum())

# Guarda le prime righe delle colonne problematiche
print(df.head())

sepal.length    float64
sepal.width     float64
petal.length    float64
petal.width     float64
variety          object
dtype: object
['variety']
sepal.length    0
sepal.width     0
petal.length    0
petal.width     0
variety         0
dtype: int64
   sepal.length  sepal.width  petal.length  petal.width variety
0           5.1          3.5           1.4          0.2  Setosa
1           4.9          3.0           1.4          0.2  Setosa
2           4.7          3.2           1.3          0.2  Setosa
3           4.6          3.1           1.5          0.2  Setosa
4           5.0          3.6           1.4          0.2  Setosa


## Step 6: Preprocessing - Encoding delle Variabili Categoriche

La colonna `variety` è categorica. La convertiamo in numeri interi (0, 1, 2).

In [6]:
df['variety'] = df['variety'].astype('category').cat.codes

## Step 7: Conversione in Tensori PyTorch

Converti il DataFrame in tensori PyTorch:
- **X**: feature (dimensioni 4) come float32
- **y**: etichette come valori interi (long)

In [7]:
import torch
# ── 3. CONVERTI IN TENSORI ───────────────────────────────────────────────────
X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)  # feature → float
y = torch.tensor(df.iloc[:,  -1].values, dtype=torch.long)     # label   → int

## Step 8: Sharding e Upload su GCS

Dividi il dataset in **4 shard** (frammenti) e carica ogni shard come file `.pt` su GCS.

Ogni shard contiene:
- `chunk['X']`: tensore con le feature dello shard
- `chunk['y']`: tensore con le etichette dello shard

In [8]:
import math

# ── 4. CREA E CARICA GLI SHARD SU GCS ────────────────────────────────────────
NUM_SHARDS = 4   # con dataset piccoli bastano pochi shard; adatta a num_workers
shard_size = math.ceil(len(X) / NUM_SHARDS)

for i in range(NUM_SHARDS):
    start = i * shard_size
    end   = min(start + shard_size, len(X))      # l'ultimo shard può essere più corto

    chunk = {
        'X': X[start:end],   # tensore [shard_size, 4]
        'y': y[start:end],   # tensore [shard_size]
    }

    uri = f'gs://{NOME_DEL_BUCKET}/shards/shard-{i:03d}.pt'
    with fs.open(uri, 'wb') as f:
        torch.save(chunk, f)

    print(f"Shard {i:03d}: righe {start}–{end-1}  ({len(chunk['X'])} campioni) → {uri}")

Shard 000: righe 0–37  (38 campioni) → gs://ias-luigi-asprino-bucket/shards/shard-000.pt
Shard 001: righe 38–75  (38 campioni) → gs://ias-luigi-asprino-bucket/shards/shard-001.pt
Shard 002: righe 76–113  (38 campioni) → gs://ias-luigi-asprino-bucket/shards/shard-002.pt
Shard 003: righe 114–149  (36 campioni) → gs://ias-luigi-asprino-bucket/shards/shard-003.pt


## Step 9: Verifica dello Shard

Leggi il primo shard da GCS e verifica le dimensioni del tensore.

In [9]:
with gcsfs.GCSFileSystem(token="google_default").open("gs://ias-luigi-asprino-bucket/shards/shard-000.pt", 'rb') as f:
    chunk = torch.load(io.BytesIO(f.read()))

print(chunk['X'].shape)
print(chunk['y'].shape)

torch.Size([38, 4])
torch.Size([38])


## Step 10: Dataset Lazy con Streaming da GCS

La classe `GCSShardDataset` implementa `torch.utils.data.Dataset`:
- **Lazy loading**: carica i dati solo quando richiesti (non all'inizializzazione)
- **Caching per worker**: ogni processo worker mantiene in memoria gli ultimi 2 shard scaricati
- **Streaming**: scarica i shard da GCS on-demand durante l'addestramento

La classe è salvata nel file `gcpbucketdataset.py` per evitare errori di pickling con `num_workers > 0`.

## Step 11: Creazione del DataLoader

Crea un `DataLoader` con:
- **batch_size**: metà della dimensione dello shard
- **num_workers**: 2 processi worker per parallelizzare il caricamento dei dati
- Ogni worker gestisce il proprio cache locale degli shard

Il loop di addestramento itera su epoch e batch, caricando i dati on-demand da GCS.

In [10]:
# Parametri — devono corrispondere a quelli usati durante lo sharding
from gcpbucketdataset import GCSShardDataset
from torch.utils.data import DataLoader

dataset = GCSShardDataset(NOME_DEL_BUCKET, NUM_SHARDS, shard_size, total_samples=len(X))

dl = DataLoader(
    dataset,
    batch_size  = shard_size // 2,
    num_workers = 2
)

# Training loop
for epoch in range(2):
    print(f"Epoch {epoch}")
    for X_batch, y_batch in dl:
        print(f"epoch: {epoch} X: {len(X_batch)} Y: {len(y_batch)}")
        # forward / backward ...


Epoch 0


I0430 07:56:42.405900   97360 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0430 07:56:42.420206   98504 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0430 07:56:42.429674   97360 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0430 07:56:42.443507   98520 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0430 07:56:42.443604   98520 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0430 07:56:42.455255   97360 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0430 07:56:42.469059   98545 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0430 07:56:47.093284   98551 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0430 07:56:47.111586   98702 ev_poll_posix.cc:593] FD from fork

epoch: 0 X: 19 Y: 19
epoch: 0 X: 19 Y: 19
epoch: 0 X: 19 Y: 19
epoch: 0 X: 19 Y: 19
epoch: 0 X: 19 Y: 19
epoch: 0 X: 19 Y: 19
epoch: 0 X: 19 Y: 19
epoch: 0 X: 17 Y: 17
Epoch 1


I0430 07:56:59.329007   97360 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0430 07:56:59.345825   98864 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0430 07:56:59.353000   97360 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0430 07:56:59.367067   98889 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0430 07:57:03.875246   98896 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0430 07:57:03.875531   98873 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0430 07:57:03.891541   99052 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(26, generation: 1)
I0430 07:57:03.891881   99065 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(26, generation: 1)


epoch: 1 X: 19 Y: 19
epoch: 1 X: 19 Y: 19
epoch: 1 X: 19 Y: 19
epoch: 1 X: 19 Y: 19
epoch: 1 X: 19 Y: 19
epoch: 1 X: 19 Y: 19
epoch: 1 X: 19 Y: 19
epoch: 1 X: 17 Y: 17
